In [ ]:
from jaxl.constants import *
from jaxl.datasets import get_dataset
from jaxl.models import load_config, iterate_models
from jaxl.utils import parse_dict, set_seed
from utils import *

import jax.numpy as jnp
import os

# Penzai
from penzai import pz

import IPython

pz.ts.register_as_default()

# Optional automatic array visualization extras:
pz.ts.register_autovisualize_magic()
pz.enable_interactive_context()
pz.ts.active_autovisualizer.set_interactive(pz.ts.ArrayAutovisualizer())

In [ ]:
seed = 42
set_seed(seed)

In [ ]:
log_dir = "/Users/chanb/research/personal/jaxl/experiments/stream_block_biuniform/logs"
# experiment_name = "dist_properties"
# variant = "1000_low_freq_clusters-07-12-24_20_13_51-2fea28dc-f66a-4ec2-b0b5-0a2c6473030d"
# variant = "50000_low_freq_clusters-07-12-24_18_44_00-54bf15ea-c79e-4c3e-b652-d293c8943b72"

experiment_name = "dist_properties-num_heads_1"
variant = "1000_low_freq_clusters-07-16-24_19_53_48-8bffdf3c-46a0-4a23-9d66-892b4837d890"

learner_path = os.path.join(log_dir, experiment_name, variant)

In [ ]:
config_dict, config = load_config(learner_path)
config_dict["learner_config"]["batch_size"] = 1
config = parse_dict(config_dict)

In [ ]:
train_dataset = get_dataset(
    config.learner_config.dataset_config,
    config.learner_config.seeds.data_seed,
)
train_datalaoder = iter(train_dataset.get_dataloader(config.learner_config))

In [ ]:
model_iter = iterate_models(
    train_dataset.input_dim, train_dataset.output_dim, learner_path
)

params_init, model, checkpoint_step_init = next(model_iter)

for _ in range(40):
    params_next, _, checkpoint_step_next = next(model_iter)

In [ ]:
for sample in train_datalaoder:
    if np.all(np.sum(sample["context_outputs"][0], axis=0) > 0) and sample["context_outputs"][0][-1][1] == 1:
        break
    # if np.any(np.sum(sample["context_outputs"][0], axis=0) == 0) and sample["context_outputs"][0][0][1] == 1:
    #     break

In [ ]:
sample["context_outputs"][0]

In [ ]:
sample["outputs"][0]

In [ ]:
result = dict()
for params, checkpoint_step in zip(
    [params_init, params_next],
    [checkpoint_step_init, checkpoint_step_next]
):
    out, _, aux = model.get_attention(
        params[CONST_MODEL_DICT][CONST_MODEL],
        sample["queries"],
        {
            CONST_CONTEXT_INPUT: sample["context_inputs"],
            CONST_CONTEXT_OUTPUT: sample["context_outputs"]
        },
        eval=True,
    )

    result[checkpoint_step] = dict()
    for block in aux["gpt"]["intermediates"]:
        if not block.startswith("GPTBlock_"):
            continue

        # axis=1 -> query
        # axis=2 -> key
        self_attention_map = aux["gpt"]["intermediates"][block]["SelfAttentionModule_0"]["attention"][0][0]
        self_attention_map = self_attention_map.at[self_attention_map <= -1e10].set(jnp.nan)

        attention_score = jnp.sum(aux["gpt"]["intermediates"][block]["attention"][0][0], axis=-1)
        input_vector = jnp.sum(aux["gpt"]["intermediates"][block]["input"][0][0], axis=-1)
        block_output = jnp.sum(aux["gpt"]["intermediates"][block]["block_out"][0][0], axis=-1)

        result[checkpoint_step][block] = dict(
            self_attention_map=self_attention_map,
            attention_score=jnp.vstack(
                (
                    jnp.concatenate((np.argmax(sample["context_outputs"][0], axis=-1), jnp.array([jnp.nan])), axis=-1),
                    attention_score[::2],
                    jnp.concatenate((attention_score[1::2], jnp.array([jnp.nan])), axis=-1),
                )
            ),
            input_vector=jnp.vstack(
                (
                    jnp.concatenate((np.argmax(sample["context_outputs"][0], axis=-1), jnp.array([jnp.nan])), axis=-1),
                    input_vector[::2],
                    jnp.concatenate((input_vector[1::2], jnp.array([jnp.nan])), axis=-1),
                )
            ),
            block_output=jnp.vstack(
                (
                    jnp.concatenate((np.argmax(sample["context_outputs"][0], axis=-1), jnp.array([jnp.nan])), axis=-1),
                    block_output[::2],
                    jnp.concatenate((block_output[1::2], jnp.array([jnp.nan])), axis=-1),
                )
            ),
        )

In [ ]:
sample["context_inputs"].shape, sample["queries"].shape

In [ ]:
np.mean((sample["context_inputs"][0] - sample["queries"][0]) ** 2, axis=-1)

In [ ]:
np.sum(sample["context_inputs"][0] * sample["queries"][0], axis=-1)

In [ ]:
pz.ts.display(result)

In [ ]:
sample["outputs"]